In [38]:
pip install openpyxl


[notice] A new release of pip is available: 26.0 -> 26.0.1
[notice] To update, run: /Users/rain/Desktop/DSP/COEQWAL_V3/.venv311/bin/python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [39]:
pip install pandas


[notice] A new release of pip is available: 26.0 -> 26.0.1
[notice] To update, run: /Users/rain/Desktop/DSP/COEQWAL_V3/.venv311/bin/python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [40]:
pip install numpy


[notice] A new release of pip is available: 26.0 -> 26.0.1
[notice] To update, run: /Users/rain/Desktop/DSP/COEQWAL_V3/.venv311/bin/python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [41]:
pip install plotly


[notice] A new release of pip is available: 26.0 -> 26.0.1
[notice] To update, run: /Users/rain/Desktop/DSP/COEQWAL_V3/.venv311/bin/python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [42]:
pip install matplotlib


[notice] A new release of pip is available: 26.0 -> 26.0.1
[notice] To update, run: /Users/rain/Desktop/DSP/COEQWAL_V3/.venv311/bin/python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [43]:
pip install seaborn


[notice] A new release of pip is available: 26.0 -> 26.0.1
[notice] To update, run: /Users/rain/Desktop/DSP/COEQWAL_V3/.venv311/bin/python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [44]:
pip install dash


[notice] A new release of pip is available: 26.0 -> 26.0.1
[notice] To update, run: /Users/rain/Desktop/DSP/COEQWAL_V3/.venv311/bin/python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [45]:
import plotly.graph_objects as go

In [46]:
# Import standard libraries
import os
from contextlib import redirect_stdout

import sys
# append coeqwal packages to path
sys.path.append('./coeqwalpackage')

import numpy as np
import pandas as pd
import datetime as dt

In [47]:
# Import custom libraries
# Note: on my computer the next import doesn't work the first time I call it, why? If I re-run the cell, then it is ok. MUST DEBUG
from coeqwalpackage.metrics import *
import cqwlutils as cu
import plotting as pu
import re
import dash
from dash import dcc, html
from dash.dependencies import Input, Output

In [48]:
CtrlFile = 'CalSim3DataExtractionInitFile_v4.xlsx'
CtrlTab = 'Init'
ScenarioListFile, ScenarioListTab, ScenarioListPath, DVDssNamesOutPath, SVDssNamesOutPath, ScenarioIndicesOutPath, DssDirsOutPath, VarListPath, VarListFile, VarListTab, VarOutPath, DataOutPath, ConvertDataOutPath, ExtractionSubPath, DemandDeliverySubPath, ModelSubPath, GroupDataDirPath, ScenarioDir, DVDssMin, DVDssMax, SVDssMin, SVDssMax, NameMin, NameMax, DirMin, DirMax, IndexMin, IndexMax, StartMin, StartMax, EndMin, EndMax, VarMin, VarMax, DemandFilePath, DemandFileName, DemandFileTab, DemMin, DemMax, InflowOutSubPath, InflowFilePath, InflowFileName, InflowFileTab, InflowMin, InflowMax = cu.read_init_file(CtrlFile, CtrlTab)

In [49]:
df, dss_names = read_in_df(ConvertDataOutPath,DVDssNamesOutPath)

In [50]:
df = add_water_year_column(df)

In [51]:
metrics_path = GroupDataDirPath + "/metrics_output"
if not os.path.exists(metrics_path):
    os.makedirs(metrics_path)

plots_path = GroupDataDirPath + "/plots_output"
if not os.path.exists(plots_path):
    os.makedirs(plots_path)

output_data_path = "./Dashboard/data"
if not os.path.exists(output_data_path):
    os.makedirs(output_data_path)

output_filename = output_data_path + "/calsim_dashboard_df.csv"
output_group_data_filename = GroupDataDirPath + "/calsim_dashboard_df.csv"

In [52]:
drought_wys = [
    1924,1925,1926,1929,1930,1931,1932,1933,1934,
    1939,1944,1945,1947,1948,1949,1950,1955,1960,
    1961,1962,1964,1976,1977,1979,1981,1987,1988,
    1989,1990,1991,1992,1994,2001,2008,2009,2013,
    2014,2015,2020,2021
]

In [53]:
df.columns = ['_'.join(map(str, col)) for col in df.columns]

print(df.columns.tolist()[:20]) 

['WaterYear______', 'CALSIM_AWOANN_64_XADV_s0001_ANNUAL-APPLIED-WATER_1MON_L2020A_PER-AVER_TAF', 'CALSIM_AWOANN_72_XA1DV_s0001_ANNUAL-APPLIED-WATER_1MON_L2020A_PER-AVER_TAF', 'CALSIM_AWOANN_72_XA2DV_s0001_ANNUAL-APPLIED-WATER_1MON_L2020A_PER-AVER_TAF', 'CALSIM_AWOANN_72_XA3DV_s0001_ANNUAL-APPLIED-WATER_1MON_L2020A_PER-AVER_TAF', 'CALSIM_AWOANN_73_XADV_s0001_ANNUAL-APPLIED-WATER_1MON_L2020A_PER-AVER_TAF', 'CALSIM_BANKSEC_MAX14DAY_s0001_SALINITY-APPROX_1MON_L2020A_PER-AVER_UMHOS/CM', 'CALSIM_COREQSACDV_s0001_FLOW_1MON_L2020A_PER-AVER_CFS', 'CALSIM_CO_EC_MONTH_s0001_SALINITY_1MON_L2020A_PER-AVER_UMHOS/CM', 'CALSIM_C_AMR004_s0001_CHANNEL_1MON_L2020A_PER-AVER_CFS', 'CALSIM_C_AMR004_ADD_s0001_FLOW-ADDITIONAL-INSTREAM_1MON_L2020A_PER-AVER_CFS', 'CALSIM_C_AMR004_MIF_s0001_FLOW-MIN-INSTREAM_1MON_L2020A_PER-AVER_CFS', 'CALSIM_C_CAA003_s0001_CHANNEL_1MON_L2020A_PER-AVER_CFS', 'CALSIM_C_CAA003_CVP_s0001_FLOW-DELIVERY_1MON_L2020A_PER-AVER_CFS', 'CALSIM_C_CAA003_SWP_s0001_FLOW-DELIVERY_1MON_L2020A_P

In [54]:
original_columns = df.columns.tolist()

s_numbers = set()
for col in original_columns:
    matches = re.findall(r's\d{4,}', col)  
    s_numbers.update(matches) 

# Convert to a sorted list
s_numbers_list = sorted(s_numbers)

print(s_numbers_list)

['s0001', 's0002', 's0003', 's0004', 's0005', 's0006', 's0007', 's0008', 's0009', 's0010', 's0011', 's0012', 's0013', 's0014', 's0015', 's0016', 's0018', 's0019', 's0020', 's0021', 's0022', 's0023', 's0024', 's0025', 's0027', 's0029', 's0046']


### Time Series
Shows how the selected variable changes over time for each scenario.

### Monthly-of-Year
Displays the monthly average for a selected year.

### Single Exceedance
Shows the probability that a value will be equaled or exceeded.

### Annual Exceedance
Shows how often the selected month's total value exceeds a given threshold across all years. Each year’s data for the chosen month is summed (e.g., total flow in April each year), and the annual values are ranked from highest to lowest. From these ranks, exceedance probabilities are calculated to show how frequently high values occur.

### Month-of-Year Avg
Averages each calendar month across all years, optionally filtered by Water Year Type. Water Year Types classify each year based on how wet or dry it was. The scale ranges from 1 (wettest) to 5 (driest)

In [55]:
import re
import numpy as np
import pandas as pd
import plotly.graph_objs as go
import dash
from dash import dcc, html
from dash.dependencies import Input, Output, State
from dash.exceptions import PreventUpdate

DATA_PATH = "data/calsim_dashboard_df.csv"
VARIABLE_DESC_PATH = "data/trend_report_variables_v5.csv"
SCENARIO_DESC_PATH = "data/coeqwal_cs3_scenario_listing_v5.csv"
VARIABLE_GROUPS_PATH = "data/variable_groupings.csv"
SCENARIO_GROUPS_PATH = "data/scenario_groupings.csv"

df = pd.read_csv(DATA_PATH, parse_dates=[0], index_col=0)
df.index = pd.to_datetime(df.index)

df_var = pd.read_csv(VARIABLE_DESC_PATH)
df_scen = pd.read_csv(SCENARIO_DESC_PATH)

def read_csv_flexible(path):
    try:
        return pd.read_csv(path, encoding="utf-8")
    except Exception:
        return pd.read_csv(path, encoding="latin1")

var_groups_raw = read_csv_flexible(VARIABLE_GROUPS_PATH)
scen_groups_raw = read_csv_flexible(SCENARIO_GROUPS_PATH)

original_columns = list(map(str, df.columns))

pat_var = re.compile(r"CALSIM_(.*?)_s\d{4}", re.IGNORECASE)
pat_s = re.compile(r"s\d{4,}", re.IGNORECASE)

records = []
for col in original_columns:
    m = pat_var.search(col)
    var = m.group(1) if m else None
    s_list = pat_s.findall(col)
    scen = s_list[0] if s_list else None
    unit = col.split("_")[-1] if "_" in col else ""
    records.append({"column": col, "variable": var, "scenario": scen, "unit": unit})

meta = pd.DataFrame(records)
scenarios = sorted({r["scenario"] for r in records if r["scenario"]})

variable_unit_pairs = []
for var, sub in meta.groupby("variable", dropna=True):
    units = {str(u).upper() for u in sub["unit"] if pd.notna(u)}
    for u in units:
        variable_unit_pairs.append((var, u))

variables = sorted([f"{v}__{u}" for v, u in variable_unit_pairs])

variable_labels = [
    {"label": f"{v} ({u})", "value": f"{v}__{u}"}
    for v, u in variable_unit_pairs
]

def add_water_year_column(df_in):
    df_copy = df_in.copy().sort_index()
    df_copy["Date"] = pd.to_datetime(df_copy.index)
    df_copy["Year"] = df_copy["Date"].dt.year
    df_copy["Month"] = df_copy["Date"].dt.month
    df_copy["WaterYear"] = np.where(df_copy["Month"] >= 10, df_copy["Year"] + 1, df_copy["Year"])
    return df_copy.drop(["Date", "Year", "Month"], axis=1)

water_year_df = add_water_year_column(df)

years_in_data = sorted(df.index.year.unique())
drought_years = {
    1924, 1925, 1926, 1929, 1930, 1931, 1932, 1933, 1934, 1939,
    1944, 1945, 1947, 1948, 1949, 1950, 1955, 1960, 1961, 1962, 1964,
    1976, 1977, 1979, 1981, 1987, 1988, 1989, 1990, 1991, 1992, 1994,
    2001, 2008, 2009, 2013, 2014, 2015, 2020, 2021
}
year_options = [{"label": f"{y} {'(Historical Drought Years)' if y in drought_years else ''}", "value": y} for y in years_in_data]
water_year_type_options = [{"label": str(i), "value": i} for i in range(1, 6)]
month_options = [{"label": name, "value": i} for i, name in enumerate(
    ["January","February","March","April","May","June","July","August","September","October","November","December"], start=1
)]

plot_type_descriptions = {
    "time_series": "Shows how the selected variable changes over time for each scenario.",
    "monthly": "Displays the monthly average across selected year(s).",
    "single_exceedance": "Shows the probability that a value will be equaled or exceeded.",
    "annual_exceedance": "Shows how often the selected month's total value exceeds a given threshold across all years.",
    "month_of_year_avg": "Averages each calendar month across all years, optionally filtered by Water Year Type."
}

def find_col(df_in, var_unit, scenario):
    var, unit = var_unit.split("__")
    suffix = f"_{unit.lower()}"
    matches = [
        c for c in df_in.columns
        if var in str(c)
        and scenario in str(c)
        and str(c).lower().endswith(suffix)
    ]
    return matches[0] if matches else None

def find_wyt_col(df_in, scenario):
    matches = [c for c in df_in.columns if f"CALSIM_WYT_SAC__{scenario}" in str(c) and "WATERYEARTYPE" in str(c)]
    return matches[0] if matches else None

def get_colors(scenarios_list):
    base = ["red", "blue", "green", "orange", "purple", "brown", "cyan", "magenta", "gray", "black"]
    return {s: base[i % len(base)] for i, s in enumerate(scenarios_list)}

def get_line_styles():
    return ["solid", "dash", "dot", "dashdot", "longdash", "longdashdot"]

def filter_by_wyt_annual(df_col, scenario, wyt_list, month=5):
    if not wyt_list:
        return df_col
    wyt_col = find_wyt_col(df, scenario)
    if wyt_col is None:
        return df_col
    working_df = df[[wyt_col]].copy()
    working_df["WaterYear"] = water_year_df["WaterYear"]
    working_df["Month"] = df.index.month
    filtered = working_df[working_df["Month"] == month].groupby("WaterYear").first()
    selected_years = filtered[filtered[wyt_col].isin(wyt_list)].index
    out = df_col.copy()
    out["WaterYear"] = water_year_df["WaterYear"]
    out = out[out["WaterYear"].isin(selected_years)]
    return out.drop(columns="WaterYear")

def normalize_scenario_token(tok):
    if tok is None:
        return None
    s = str(tok).strip()
    if not s:
        return None
    m = re.match(r"^s(\d+)$", s, flags=re.IGNORECASE)
    if m:
        n = int(m.group(1))
        return f"s{n:04d}"
    if re.match(r"^\d+$", s):
        n = int(s)
        return f"s{n:04d}"
    m2 = re.search(r"(s\d+)", s, flags=re.IGNORECASE)
    if m2:
        return normalize_scenario_token(m2.group(1))
    return None

def parse_unit_from_group_name(text):
    m = re.search(r"\(([^)]+)\)", str(text) if text is not None else "")
    return m.group(1).strip().upper() if m else None

def split_tokens(text):
    if text is None:
        return []
    s = str(text)
    parts = re.split(r"[,\n;|]+", s)
    return [p.strip() for p in parts if p and p.strip()]

def clean_var_token(token):
    if token is None:
        return None
    t = str(token).strip()
    if not t:
        return None
    t = re.sub(r"\s*\([^)]*\)\s*$", "", t).strip()
    return t if t else None

def unit_in_token(token):
    if token is None:
        return None
    m = re.search(r"\(([^)]+)\)\s*$", str(token).strip())
    return m.group(1).strip().upper() if m else None

def build_variable_groups(df_groups):
    colmap = {str(c).strip().lower(): c for c in df_groups.columns}
    c_group = colmap.get("grouping")
    c_desc = colmap.get("description")
    c_vars = colmap.get("variables")
    if c_group is None or c_vars is None:
        return {}
    groups = {}
    for _, row in df_groups.iterrows():
        name = str(row.get(c_group, "")).strip()
        if not name:
            continue
        desc = str(row.get(c_desc, "")).strip() if c_desc else ""
        unit = parse_unit_from_group_name(name)
        vars_raw = row.get(c_vars, "")
        tokens = split_tokens(vars_raw)
        cleaned = []
        for tok in tokens:
            v = clean_var_token(tok)
            if v:
                cleaned.append(tok.strip())
        groups[name] = {"description": desc, "unit": unit, "variables": cleaned}
    return groups

def build_scenario_groups(df_groups):
    colmap = {str(c).strip().lower(): c for c in df_groups.columns}
    c_group = colmap.get("grouping")
    c_desc = colmap.get("description")
    c_sc = colmap.get("scenarios")
    if c_group is None or c_sc is None:
        return {}
    groups = {}
    for _, row in df_groups.iterrows():
        name = str(row.get(c_group, "")).strip()
        if not name:
            continue
        desc = str(row.get(c_desc, "")).strip() if c_desc else ""
        sc_raw = row.get(c_sc, "")
        tokens = split_tokens(sc_raw)
        norm = []
        for tok in tokens:
            ns = normalize_scenario_token(tok)
            if ns:
                norm.append(ns)
        groups[name] = {"description": desc, "scenarios": norm}
    return groups

var_groups_store_init = build_variable_groups(var_groups_raw)
scen_groups_store_init = build_scenario_groups(scen_groups_raw)

available_varunit_values = {f"{v}__{u.upper()}" for v, u in variable_unit_pairs}

def group_options(store):
    return [{"label": k, "value": k} for k in sorted(store.keys())]


app = dash.Dash(__name__)
server = app.server

app.layout = html.Div(
    style={"fontFamily": "Inter, Arial, sans-serif", "padding": "30px"},
    children=[

        dcc.Store(id="var-groups-store", data=var_groups_store_init),
        dcc.Store(id="scen-groups-store", data=scen_groups_store_init),

        html.Img(
            src="assets/coeqwal_logo_outlines.png",
            style={'height': '80px', 'display': 'block', 'marginLeft': 'auto', 'marginRight': 'auto'}
        ),
        
        html.H1(
            "Water Data Dashboard",
            style={"textAlign": "center", "fontWeight": "800", "fontSize": "30px", "marginBottom": "40px", "color": "#135773"},
        ),

        html.Div(
            style={"maxWidth": "900px", "margin": "0 auto", "backgroundColor": "white",
                   "padding": "30px", "borderRadius": "16px", "boxShadow": "0 10px 30px rgba(0,0,0,0.08)"},
            children=[
                    html.Div(style={"display": "flex", "gap": "30px", "marginBottom": "25px"},
                        children=[
                            html.Div(style={"flex": "1"},
                                children=[
                                    html.Label(
                                        "Variable Groups",
                                        style={"fontWeight": "600", "marginBottom": "0px","display": "block"}),
                                    dcc.Dropdown(
                                        id="var-group-dropdown",
                                        options=[{"label": k, "value": k}for k in sorted(var_groups_store_init.keys())],
                                        placeholder="Choose variable group..."),
                                    html.H5(
                                        id="var-group-desc",
                                        children="Variable Group Represents:",
                                        style={"marginTop": "0px", "marginBottom": "0px", "fontWeight": "500","color": "#5A5A5A"})]),
                            html.Div(
                                style={"flex": "1"},
                                children=[
                                    html.Label(
                                        "Scenario Groups",
                                        style={"fontWeight": "600", "marginBottom": "0px", "display": "block"}),
                                    dcc.Dropdown(
                                        id="scen-group-dropdown",
                                        options=[{"label": k, "value": k} for k in sorted(scen_groups_store_init.keys())],
                                        placeholder="Choose scenario group..."),
                                    html.H5(
                                        id="scen-group-desc",
                                        children="Scenario Group Represents:",
                                        style={"marginTop": "0px", "marginBottom": "0px", "fontWeight": "500", "color": "#5A5A5A"})])]),
                html.Div(style={"marginBottom": "25px"}, children=[
                    html.Label("Select Variables (must have the same units)", style={"fontWeight": "600", "marginBottom": "0px", "display": "block"}),
                    dcc.Dropdown(id="variable-dropdown", options=variable_labels,
                                 value=[], multi=True,
                                 placeholder="Choose variables..."),
                    html.H5(id='variable-defs', children="Variable Represents:", style={"marginTop": "0px", "marginBottom": "0px", "fontWeight": "500", "color": "#5A5A5A"}),
                ]),
                html.Div(style={"marginBottom": "25px"}, children=[
                    html.Label("Select Scenarios", style={"fontWeight": "600", "marginBottom": "0px", "display": "block"}),
                    dcc.Dropdown(id="scenario-dropdown",
                                 options=[{"label": s, "value": s} for s in scenarios],
                                 value=[], multi=True,
                                 placeholder="Choose scenarios..."),
                    html.H5(id='scenario-defs', children="Scenario Represents:", style={"marginTop": "0px", "marginBottom": "0px", "fontWeight": "500", "color": "#5A5A5A"}),
                ]),
                html.Div(style={"marginBottom": "25px"}, children=[
                    html.Label("Select Water Year Types (1–5)", style={"fontWeight": "600", "marginBottom": "0px", "display": "block"}),
                    dcc.Dropdown(id="wyt-dropdown",
                                 options=[{"label": i, "value": i} for i in range(1, 6)],
                                 value=None, multi=True,
                                 placeholder="Choose water year types..."),
                    html.H5("Note: 1 = wettest, 5 = driest",  style={"marginTop": "0px", "marginBottom": "0px", "fontWeight": "500", "color": "#5A5A5A"}),
                ]),
                html.Div(style={"marginBottom": "25px"}, children=[
                    html.Label("Select Year (Month-of-Year Average Plot)", style={"fontWeight": "600", "marginBottom": "6px", "display": "block"}),
                    dcc.Dropdown(id="year-dropdown",
                                 options=[{"label": y, "value": y} for y in sorted(df.index.year.unique())],
                                 value=None, placeholder="Choose year...", clearable=True),
                ]),
                html.Div(style={"marginBottom": "25px"}, children=[
                    html.Label("Select Month (Monthly Exceedance Plot)", style={"fontWeight": "600", "marginBottom": "6px", "display": "block"}),
                    dcc.Dropdown(id="month-dropdown",
                                 options=[{"label": "April", "value": 4}, {"label": "September", "value": 9}],
                                 value=None, placeholder="Choose month...", clearable=True),
                ]),
            ]
        ),

        html.Div(id="plots-container", style={"fontFamily": "Inter, Arial, sans-serif", "paddingTop": "30px"},
            children=[

            html.Div(
                style={"backgroundColor": "white", "padding": "20px", "marginTop": "20px",
                       "borderRadius": "12px", "boxShadow": "0 5px 15px rgba(0,0,0,0.05)"},
                children=[
                    html.H3("Time Series", style={"marginBottom": "6px", "color": "black"}),
                    html.P("Shows how the selected variable changes over time for each scenario", style={"marginTop": "0px", "marginBottom": "0px", "color": "black"}),
                    dcc.Graph(id="fig-time"),
                ]
            ),

            html.Div(
                style={"backgroundColor": "white", "padding": "20px", "marginTop": "20px",
                       "borderRadius": "12px", "boxShadow": "0 5px 15px rgba(0,0,0,0.05)"},
                children=[
                    html.H3("Month-of-Year Average (WYT: All)", style={"marginBottom": "6px", "color": "black"}),
                    html.P("Averages each calendar month across all years", style={"marginTop": "0px", "marginBottom": "0px", "color": "black"}),
                    dcc.Graph(id="fig-moy"),
                ]
            ),

            html.Div(
                style={"backgroundColor": "white", "padding": "20px", "marginTop": "20px",
                       "borderRadius": "12px", "boxShadow": "0 5px 15px rgba(0,0,0,0.05)"},
                children=[
                    html.H3(id="title-monthly", children="Month-of-Year Average (WYT: All)", style={"marginBottom": "6px", "color": "black"}),
                    html.P("Averages each calendar month across all years, filtered by water year type", style={"marginTop": "0px", "marginBottom": "0px", "color": "black"}),
                    dcc.Graph(id="fig-moy-wyt"),
                ]
            ),

            html.Div(
                style={"backgroundColor": "white", "padding": "20px", "marginTop": "20px",
                       "borderRadius": "12px", "boxShadow": "0 5px 15px rgba(0,0,0,0.05)"},
                children=[
                    html.H3(id="title-moy", children="Month-of-Year Average (Year: All)", style={"marginBottom": "6px", "color": "black"}),
                    html.P("Displays the monthly average for a selected year", style={"marginTop": "0px", "marginBottom": "0px", "color": "black"}),
                    dcc.Graph(id="fig-moy-year"),
                ]
            ),

            html.Div(
                style={"backgroundColor": "white", "padding": "20px", "marginTop": "20px",
                       "borderRadius": "12px", "boxShadow": "0 5px 15px rgba(0,0,0,0.05)"},
                children=[
                    html.H3("Annual Exceedance", style={"marginBottom": "6px", "color": "black"}),
                    html.P("Shows the probability that a value will be equaled or exceeded", style={"marginTop": "0px", "marginBottom": "0px", "color": "black"}),
                    dcc.Graph(id="fig-single"),
                ]
            ),

            html.Div(
                style={"backgroundColor": "white", "padding": "20px", "marginTop": "20px",
                       "borderRadius": "12px", "boxShadow": "0 5px 15px rgba(0,0,0,0.05)"},
                children=[
                    html.H3(id="title-annual", children="Monthly Exceedance (Month: All)", style={"marginBottom": "6px", "color": "black"}),
                    html.P("Shows how often the selected month's total value exceeds a given threshold across all years", style={"marginTop": "0px", "marginBottom": "0px", "color": "black"}),
                    dcc.Graph(id="fig-annual"),
                ]
            ),

        ])
    ]
)

@app.callback(
    Output("var-group-desc", "children"),
    Input("var-group-dropdown", "value"),
    State("var-groups-store", "data")
)
def show_var_group_desc(group_name, store):
    if not group_name:
        return 'Select Group'
    g = store[group_name]
    return f"{g['description']},  Units: {g['unit']}"

@app.callback(
    Output("scen-group-desc", "children"),
    Input("scen-group-dropdown", "value"),
    State("scen-groups-store", "data")
)
def show_scen_group_desc(group_name, store):
    if not group_name:
        return 'Select Group'
    return store[group_name]["description"]

@app.callback(
    Output("variable-dropdown", "options"),
    Input("var-group-dropdown", "value"),
    State("var-groups-store", "data"),
)
def filter_variables_by_group(group_name, store):

    if not group_name:
        # Show all if no group selected
        return variable_labels

    group = store[group_name]

    filtered = []

    for raw in group["variables"]:
        vname = clean_var_token(raw)
        unit = unit_in_token(raw) or group["unit"]

        if not vname or not unit:
            continue

        val = f"{vname}__{unit}"
        if val in available_varunit_values:
            filtered.append({
                "label": f"{vname} ({unit})",
                "value": val
            })

    return filtered

@app.callback(
    Output("scenario-dropdown", "options"),
    Input("scen-group-dropdown", "value"),
    State("scen-groups-store", "data"),
)
def filter_scenarios_by_group(group_name, store):

    if not group_name:
        return [{"label": s, "value": s} for s in scenarios]

    group = store[group_name]

    filtered = [
        {"label": s, "value": s}
        for s in group["scenarios"]
        if s in scenarios
    ]

    return filtered

@app.callback(
    [
        Output("variable-defs", "children"),
        Output("scenario-defs", "children"),
        Output("fig-time", "figure"),
        Output("fig-moy", "figure"),
        Output("fig-moy-wyt", "figure"),
        Output("fig-moy-year", "figure"),
        Output("fig-single", "figure"),
        Output("fig-annual", "figure"),
        Output("title-monthly", "children"),
        Output("title-moy", "children"),
        Output("title-annual", "children"),
    ],
    [
        Input("variable-dropdown", "value"),
        Input("scenario-dropdown", "value"),
        Input("year-dropdown", "value"),
        Input("wyt-dropdown", "value"),
        Input("month-dropdown", "value"),
    ]
)
def update_plots(vars_selected, scen_list, year, wyt, month):

    wyt_label = ", ".join(map(str, wyt)) if wyt else "Not Selected"
    year_label = str(year) if year else "Not Selected"
    month_label = str(month) if month else "Not Selected"

    title_monthly = f"Month-of-Year Average (WYT: {wyt_label})"
    title_moy = f"Month-of-Year Average (Year: {year_label})"
    title_annual = f"Monthly Exceedance (Month: {month_label})"

    # VARIABLE DESCRIPTIONS
    clean_vars = [v.split("__")[0] for v in vars_selected] if vars_selected else []
    
    variable_defs = (
        df_var[df_var["Unnamed: 3"].isin(clean_vars)]
        .set_index("Unnamed: 3")["Unnamed: 10"]
        .to_dict()
    )

    variable_desc_children = [
        html.Div([
            html.B(f"{var}: "),
            html.Span(desc)
        ]) for var, desc in variable_defs.items()
    ] if vars_selected else "Select Variables"

    # SCENARIO DESCRIPTIONS
    scenario_defs = (
        df_scen[df_scen["Index"].isin(scen_list)]
        .set_index("Index")["ShortDescription"]
        .to_dict()
    )

    scenario_desc_children = [
        html.Div([
            html.B(f"{scen}: "),
            html.Span(desc)
        ]) for scen, desc in scenario_defs.items()
    ] if scen_list else "Select Scenarios"
    
    def empty(title=""):
        fig = go.Figure()
        fig.update_layout(title=title)
        return fig

    if not vars_selected or not scen_list:
        return (
            variable_desc_children,
            scenario_desc_children,
            empty(""),
            empty(""),
            empty(""),
            empty(""),
            empty(""),
            empty(""),
            title_monthly,
            title_moy,
            title_annual,
        )

    def empty(title):
        fig = go.Figure()
        fig.update_layout(title=title)
        return fig

    if not vars_selected or not scen_list:
        return (
            empty(""),
            empty(""),
            empty(""),
            empty(""),
            empty(""),
            empty(""),
        )

    selected_units = {v.split("__")[1] for v in vars_selected}
    if len(selected_units) > 1:
        fig = empty("Unit mismatch — variables must match")
        return fig, fig, fig, fig, fig

    unit = selected_units.pop()
    colors = get_colors(scen_list)
    styles = get_line_styles()

    fig_ts = go.Figure()
    fig_moy = go.Figure()
    fig_moy_wyt = go.Figure()
    fig_moy_year = go.Figure()
    fig_single = go.Figure()
    fig_annual = go.Figure()

    for i, v in enumerate(vars_selected):
        style = styles[i % len(styles)]

        for s in scen_list:
            col = find_col(df, v, s)
            if not col:
                continue

            # Time Series
            df_copy = df[[col]].copy()

            fig_ts.add_trace(go.Scatter(
                x=df.index,
                y=df_copy[col],
                mode="lines",
                name=f"{s} – {v}",
                line=dict(color=colors[s], dash=style),
            ))

            # MOY
            df_m = df_copy.copy()
            df_m["Month"] = df_m.index.month

            monthly_avg = (
                df_m
                .groupby("Month")[col]
                .mean()
            )

            fig_moy.add_trace(go.Scatter(
                x=monthly_avg.index,
                y=monthly_avg.values,
                mode="lines+markers",
                name=f"{s} – {v}",
                line=dict(color=colors[s], dash=style),
            ))

            # MOY WYT
            if not wyt:
                moy_avg = pd.Series(dtype=float)
            else:
                df_sel = water_year_df[[col]].copy()
                df_sel = filter_by_wyt_annual(df_sel, s, wyt, month=5)
                df_sel["Month"] = df_sel.index.month
                moy_avg = df_sel.groupby("Month")[col].mean()

            fig_moy_wyt.add_trace(go.Scatter(
                x=moy_avg.index,
                y=moy_avg.values,
                mode="lines+markers",
                name=f"{s} – {v}",
                line=dict(color=colors[s], dash=style),
            ))

            # MOY Year
            df_m = df_copy.copy()
            df_m["Year"] = df_m.index.year
            df_m["Month"] = df_m.index.month

            monthly_avg = (
                df_m[df_m["Year"] == year]
                .groupby("Month")[col]
                .mean()
            )

            fig_moy_year.add_trace(go.Scatter(
                x=monthly_avg.index,
                y=monthly_avg.values,
                mode="lines+markers",
                name=f"{s} – {v}",
                line=dict(color=colors[s], dash=style),
            ))
            
            # Annual Exceedance
            series = df_copy[col].dropna().sort_values(ascending=False)
            exceedance_probs = np.arange(1, len(series) + 1) / (len(series) + 1)

            fig_single.add_trace(go.Scatter(
                x=exceedance_probs,
                y=series.values,
                mode="lines",
                name=f"{s} – {v}",
                line=dict(color=colors[s], dash=style),
            ))

            # Monthly Exceedance
            df_a = df_copy[df_copy.index.month == month]
            annual_sum = df_a.resample("YE").sum(min_count=1)
            sorted_vals = annual_sum[col].dropna().sort_values(ascending=False)
            exceed_probs = (
                sorted_vals.rank(method="first", ascending=False)
                / (1 + len(sorted_vals))
            )

            fig_annual.add_trace(go.Scatter(
                x=exceed_probs,
                y=sorted_vals,
                mode="lines",
                name=f"{s} – {v}",
                line=dict(color=colors[s], dash=style),
            ))

    fig_ts.update_layout(xaxis_title="Year", yaxis_title=f"Value ({unit})")
    fig_moy.update_layout(xaxis_title="Month", yaxis_title=f"Value ({unit})")
    fig_moy_wyt.update_layout(xaxis_title="Month", yaxis_title=f"Value ({unit})")
    fig_moy_year.update_layout(xaxis_title="Month", yaxis_title=f"Value ({unit})")
    fig_single.update_layout(xaxis_title="Exceedance Probability", yaxis_title=f"Value ({unit})")
    fig_annual.update_layout(xaxis_title="Exceedance Probability", yaxis_title=f"Value ({unit})")

    return variable_desc_children, scenario_desc_children, fig_ts, fig_moy, fig_moy_wyt, fig_moy_year, fig_single, fig_annual, title_monthly, title_moy, title_annual

if __name__ == "__main__":
    app.run(debug=True, host="127.0.0.1", port=8060)



### Save data

In [ ]:
df.to_csv(output_filename)
df.to_csv(output_group_data_filename)
print("Data saved in " + output_filename + " and " + output_group_data_filename)